# Legion Mobile Sandbox
**demo x hexa | Team Death Legion**

Android 15 cloud emulator — virtual SIM card, ADB, noVNC browser access.

---

**Before you run:**
1. Get a free ngrok token: https://dashboard.ngrok.com/signup
2. Paste it in **Cell 8** — `NGROK_AUTH_TOKEN = 'your_token'`
3. Set your virtual phone number in **Cell 8** — `SIM_PHONE_NUMBER`
4. **Runtime > Run all**
5. Wait ~4 minutes → Cell 10 prints the public browser URL

---

**SIM card quick reference (run after boot):**
```
adb emu sms send +15559876543 "Hello from Legion"
adb emu gsm call +15559876543
adb emu gsm signal 4 4
```
Full reference in Cell 16.

---

In [ ]:
# ─────────────────────────────────────────────
# CELL 1  System check
# ─────────────────────────────────────────────
import os, subprocess, sys

def sh(cmd, check=True, capture=False):
    r = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if check and r.returncode != 0:
        err = r.stderr if capture else ''
        raise RuntimeError(f'Command failed: {cmd}\n{err}')
    return r

print('=== Runtime info ===')
sh('uname -a')
sh('nproc')
sh('free -h')
sh('df -h /')

kvm = os.path.exists('/dev/kvm')
print(f'\nKVM: {"available — hardware acceleration ON" if kvm else "not available — software rendering (still works)"}')
print()
print('Android 15 (API 35) needs ~4 GB RAM.')
print('GPU runtime recommended: Runtime > Change runtime type > T4 GPU')
print('\nCell 1 done.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 2  System packages
# ─────────────────────────────────────────────
import subprocess

pkgs = ' '.join([
    'openjdk-17-jdk', 'xvfb', 'x11vnc', 'websockify',
    'wget', 'unzip', 'curl',
    'libgles2-mesa', 'libpulse0', 'libasound2',
    'libxcomposite1', 'libxcursor1', 'libxi6',
    'libxrandr2', 'libxss1', 'libxtst6',
    'libgl1-mesa-glx', 'libgl1-mesa-dri',
    'xauth', 'x11-xkb-utils', 'xfonts-base', 'xterm',
    'socat',  # needed for emulator console port forwarding
])

subprocess.run(f'apt-get update -qq && apt-get install -y -qq {pkgs}',
               shell=True, check=True)

subprocess.run('java -version', shell=True)
print('Cell 2 done.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 3  Android SDK command-line tools
# ─────────────────────────────────────────────
import os, subprocess

ANDROID_HOME = '/opt/android-sdk'
SDK_URL = 'https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip'
SDK_ZIP = '/tmp/cmdline-tools.zip'

os.makedirs(f'{ANDROID_HOME}/cmdline-tools', exist_ok=True)

print('Downloading SDK tools...')
subprocess.run(f'wget -q "{SDK_URL}" -O {SDK_ZIP}', shell=True, check=True)
subprocess.run(f'unzip -q {SDK_ZIP} -d {ANDROID_HOME}/cmdline-tools/', shell=True, check=True)

extracted = f'{ANDROID_HOME}/cmdline-tools/cmdline-tools'
latest    = f'{ANDROID_HOME}/cmdline-tools/latest'
if os.path.isdir(extracted) and not os.path.isdir(latest):
    os.rename(extracted, latest)

os.environ['ANDROID_HOME']     = ANDROID_HOME
os.environ['ANDROID_SDK_ROOT'] = ANDROID_HOME
os.environ['PATH'] = (
    f'{ANDROID_HOME}/cmdline-tools/latest/bin:'
    f'{ANDROID_HOME}/emulator:'
    f'{ANDROID_HOME}/platform-tools:'
    f'{ANDROID_HOME}/tools/bin:'
    + os.environ.get('PATH', '')
)

with open('/etc/profile.d/android.sh', 'w') as f:
    f.write(f'export ANDROID_HOME={ANDROID_HOME}\n')
    f.write(f'export ANDROID_SDK_ROOT={ANDROID_HOME}\n')
    f.write(f'export PATH={ANDROID_HOME}/cmdline-tools/latest/bin:'
            f'{ANDROID_HOME}/emulator:{ANDROID_HOME}/platform-tools:'
            f'{ANDROID_HOME}/tools/bin:$PATH\n')

r = subprocess.run('sdkmanager --version', shell=True, capture_output=True, text=True)
print(f'sdkmanager: {r.stdout.strip() or r.stderr.strip()}')
print('Cell 3 done.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 4  Accept licenses + install Android 15 emulator & system image
# ─────────────────────────────────────────────
import subprocess, os

# Android 15 = API level 35
API_LEVEL = '35'
ABI       = 'x86_64'
TARGET    = 'google_apis'

print('Accepting licenses...')
subprocess.run('yes | sdkmanager --licenses', shell=True, env=os.environ)

packages = [
    'platform-tools',
    'emulator',
    f'platforms;android-{API_LEVEL}',
    f'system-images;android-{API_LEVEL};{TARGET};{ABI}',
]

for pkg in packages:
    print(f'Installing {pkg}...')
    r = subprocess.run(
        f'yes | sdkmanager "{pkg}"',
        shell=True, env=os.environ, capture_output=True, text=True
    )
    if r.returncode == 0:
        print('  OK')
    else:
        print(f'  WARN: {r.stderr[-300:]}')

emu = f'{os.environ["ANDROID_HOME"]}/emulator/emulator'
assert os.path.isfile(emu), f'emulator binary missing: {emu}'
subprocess.run(f'{emu} -version', shell=True)

# Store for use in later cells
os.environ['LEGION_API_LEVEL'] = API_LEVEL
os.environ['LEGION_ABI']       = ABI
os.environ['LEGION_TARGET']    = TARGET

print('Cell 4 done.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 5  Create AVD with SIM card + telephony
# ─────────────────────────────────────────────
import subprocess, os

AVD_NAME   = 'LegionDevice'
API_LEVEL  = os.environ.get('LEGION_API_LEVEL', '35')
ABI        = os.environ.get('LEGION_ABI', 'x86_64')
TARGET     = os.environ.get('LEGION_TARGET', 'google_apis')

print(f'Creating AVD: {AVD_NAME} (Android {API_LEVEL})')
r = subprocess.run(
    f'echo no | avdmanager create avd '
    f'-n "{AVD_NAME}" '
    f'-k "system-images;android-{API_LEVEL};{TARGET};{ABI}" '
    f'--device "pixel_8" --force',
    shell=True, env=os.environ, capture_output=True, text=True
)
print(r.stdout)
if r.returncode != 0:
    print('WARN:', r.stderr[-400:])

subprocess.run('avdmanager list avd', shell=True, env=os.environ)

avd_dir    = os.path.expanduser(f'~/.android/avd/{AVD_NAME}.avd')
config_ini = os.path.join(avd_dir, 'config.ini')

# Hardware config — includes full telephony/SIM card settings
hw_overrides = {
    # CPU / RAM
    'hw.ramSize': '4096',
    'hw.cpu.ncore': '4',
    'disk.dataPartition.size': '8192M',
    'vm.heapSize': '512',
    # Display
    'hw.lcd.density': '420',
    'hw.lcd.width': '1080',
    'hw.lcd.height': '2400',
    # GPU
    'hw.gpu.enabled': 'yes',
    'hw.gpu.mode': 'swiftshader_indirect',
    # Audio
    'hw.audioInput': 'no',
    'hw.audioOutput': 'no',
    # Sensors
    'hw.battery': 'yes',
    'hw.gps': 'yes',
    'hw.sensors.proximity': 'yes',
    'hw.sensors.acceleration': 'yes',
    'hw.sensors.gyroscope': 'yes',
    'hw.sensors.magnetic_field': 'yes',
    # Camera
    'hw.camera.front': 'none',
    'hw.camera.back': 'none',
    # ── SIM card / Telephony ──────────────────────────────────
    'hw.telephony': 'yes',
    'hw.telephony.gsm': 'yes',
    'telephony.simCount': '1',
    'hw.gsm.network': 'home',
    'gsm.sim.operator.numeric': '310260',
    'gsm.sim.operator.alpha': 'T-Mobile',
    'gsm.sim.operator.iso-country': 'us',
    'gsm.defaultpdpcontext.active': 'true',
    'gsm.network.type': 'LTE',
    'gsm.version.ril-impl': 'android reference-ril 1.0',
}

if os.path.isfile(config_ini):
    with open(config_ini) as f:
        current = {}
        for line in f:
            line = line.strip()
            if '=' in line and not line.startswith('#'):
                k, v = line.split('=', 1)
                current[k] = v
    current.update(hw_overrides)
    with open(config_ini, 'w') as f:
        for k, v in current.items():
            f.write(f'{k}={v}\n')
    print('AVD config updated (SIM card + telephony enabled).')
else:
    print(f'WARN: config.ini not found at {config_ini}')

print('Cell 5 done.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 6  Install noVNC
# ─────────────────────────────────────────────
import subprocess, os

subprocess.run(
    'wget -q "https://github.com/novnc/noVNC/archive/refs/tags/v1.4.0.tar.gz" -O /tmp/novnc.tar.gz',
    shell=True, check=True
)
subprocess.run(
    'mkdir -p /opt/novnc && tar -xzf /tmp/novnc.tar.gz -C /opt/novnc --strip-components=1',
    shell=True, check=True
)
subprocess.run(
    'wget -q "https://github.com/novnc/websockify/archive/refs/tags/v0.11.0.tar.gz" -O /tmp/websockify.tar.gz',
    shell=True, check=True
)
subprocess.run(
    'mkdir -p /opt/novnc/utils/websockify && '
    'tar -xzf /tmp/websockify.tar.gz -C /opt/novnc/utils/websockify --strip-components=1',
    shell=True, check=True
)

if os.path.isfile('/opt/novnc/vnc.html') and not os.path.isfile('/opt/novnc/index.html'):
    os.symlink('/opt/novnc/vnc.html', '/opt/novnc/index.html')

print('noVNC installed at /opt/novnc')
print('Cell 6 done.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 7  Install ngrok
# ─────────────────────────────────────────────
import subprocess

subprocess.run(
    'wget -q "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz" -O /tmp/ngrok.tgz',
    shell=True, check=True
)
subprocess.run(
    'tar -xzf /tmp/ngrok.tgz -C /usr/local/bin/ && chmod +x /usr/local/bin/ngrok',
    shell=True, check=True
)

r = subprocess.run('ngrok version', shell=True, capture_output=True, text=True)
print(r.stdout.strip())
print()
print('Get your free token at: https://dashboard.ngrok.com/signup')
print('Paste it in Cell 8.')
print('Cell 7 done.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 8  Config  ← edit this cell
# ─────────────────────────────────────────────

# ── ngrok ──────────────────────────────────────────────────
# Free token at: https://dashboard.ngrok.com/signup
NGROK_AUTH_TOKEN = ''   # e.g. '2abc123xyz_YourActualTokenHere'

# ── VNC ────────────────────────────────────────────────────
VNC_PASSWORD = ''   # leave empty for no password

# ── SIM card ───────────────────────────────────────────────
# This is the virtual phone number assigned to the emulator SIM.
# Use any number — it only affects what shows in the device's dialer
# and what number appears when you use 'adb emu sms send'.
SIM_PHONE_NUMBER = '+15551234567'   # change to any number you want

# ── Network ────────────────────────────────────────────────
# Options: 'gsm', 'edge', '3g', 'hsdpa', 'lte', '5gnr'
SIM_NETWORK_TYPE = 'lte'

# Signal strength 0-4 (0 = no signal, 4 = excellent)
SIM_SIGNAL = 4

# ── Display / Ports ────────────────────────────────────────
DISPLAY_NUM = 1
VNC_PORT    = 5900 + DISPLAY_NUM
NOVNC_PORT  = 6080
SCREEN_W    = 1280
SCREEN_H    = 800
SCREEN_D    = 24

print('Config:')
print(f'  DISPLAY      : :{DISPLAY_NUM}')
print(f'  noVNC port   : {NOVNC_PORT}')
print(f'  ngrok set    : {bool(NGROK_AUTH_TOKEN)}')
print(f'  SIM number   : {SIM_PHONE_NUMBER}')
print(f'  Network type : {SIM_NETWORK_TYPE.upper()}')
print(f'  Signal       : {SIM_SIGNAL}/4')

In [ ]:
# ─────────────────────────────────────────────
# CELL 9  Start everything
# ─────────────────────────────────────────────
import subprocess, os, time, json

ANDROID_HOME = os.environ.get('ANDROID_HOME', '/opt/android-sdk')
AVD_NAME     = 'LegionDevice'
kvm_flag     = '-accel on' if os.path.exists('/dev/kvm') else '-accel off'
os.environ['DISPLAY'] = f':{DISPLAY_NUM}'

# 1. Xvfb
print('Starting Xvfb...')
subprocess.Popen(
    f'Xvfb :{DISPLAY_NUM} -screen 0 {SCREEN_W}x{SCREEN_H}x{SCREEN_D} -ac +extension RANDR',
    shell=True
)
time.sleep(2)

# 2. x11vnc
print('Starting x11vnc...')
vnc_cmd = (
    f'x11vnc -display :{DISPLAY_NUM} -forever -shared '
    f'-rfbport {VNC_PORT} -noxdamage -noxfixes '
    + (f'-passwd "{VNC_PASSWORD}" ' if VNC_PASSWORD else '-nopw ')
    + '-bg -o /tmp/x11vnc.log'
)
subprocess.run(vnc_cmd, shell=True)
time.sleep(2)

# 3. noVNC
print(f'Starting noVNC on port {NOVNC_PORT}...')
subprocess.Popen(
    f'python3 /opt/novnc/utils/websockify/run '
    f'--web /opt/novnc {NOVNC_PORT} localhost:{VNC_PORT}',
    shell=True,
    stdout=open('/tmp/novnc.log', 'w'),
    stderr=subprocess.STDOUT
)
time.sleep(2)

# 4. ngrok
public_url = None
if NGROK_AUTH_TOKEN:
    print('Starting ngrok...')
    subprocess.run(f'ngrok config add-authtoken {NGROK_AUTH_TOKEN}', shell=True, check=True)
    subprocess.Popen(f'ngrok http {NOVNC_PORT} --log /tmp/ngrok.log', shell=True)
    time.sleep(5)
    try:
        r = subprocess.run('curl -s http://localhost:4040/api/tunnels',
                           shell=True, capture_output=True, text=True)
        data = json.loads(r.stdout)
        for t in data.get('tunnels', []):
            if t.get('proto') == 'https':
                public_url = t['public_url']
                break
        if not public_url and data.get('tunnels'):
            public_url = data['tunnels'][0]['public_url']
    except Exception as e:
        print(f'WARN: could not get ngrok URL — {e}')

# 5. Android 15 emulator with SIM card
#    -phone-number   : assigns this number to the virtual SIM
#    Emulator console on port 5554 for 'adb emu' commands
print('Starting Android 15 emulator with virtual SIM...')
emu_cmd = (
    f'{ANDROID_HOME}/emulator/emulator '
    f'-avd {AVD_NAME} '
    f'{kvm_flag} '
    f'-gpu swiftshader_indirect '
    f'-no-snapshot '
    f'-no-audio '
    f'-no-boot-anim '
    f'-wipe-data '
    f'-skin 1080x2400 '
    f'-memory 4096 '
    f'-cores 4 '
    f'-phone-number {SIM_PHONE_NUMBER} '
    f'-display :{DISPLAY_NUM}'
)
emu_proc = subprocess.Popen(
    emu_cmd,
    shell=True,
    env=os.environ,
    stdout=open('/tmp/emulator.log', 'w'),
    stderr=subprocess.STDOUT
)
print(f'Emulator PID: {emu_proc.pid}')

# Store for later cells
os.environ['LEGION_EMU_PHONE'] = SIM_PHONE_NUMBER
os.environ['LEGION_NET_TYPE']  = SIM_NETWORK_TYPE
os.environ['LEGION_SIGNAL']    = str(SIM_SIGNAL)

print()
print('=' * 60)
print('  LEGION MOBILE SANDBOX — ANDROID 15 — STARTING')
if public_url:
    suffix = f'?password={VNC_PASSWORD}' if VNC_PASSWORD else ''
    print(f'  URL       : {public_url}/vnc.html{suffix}')
else:
    print(f'  LOCAL     : http://localhost:{NOVNC_PORT}/vnc.html')
    if not NGROK_AUTH_TOKEN:
        print('  (set NGROK_AUTH_TOKEN in Cell 8 for a public URL)')
print(f'  SIM number: {SIM_PHONE_NUMBER}')
print(f'  Network   : {SIM_NETWORK_TYPE.upper()}')
print('  Boot takes ~4 minutes on Colab. Run Cell 10 to wait.')
print('=' * 60)

In [ ]:
# ─────────────────────────────────────────────
# CELL 10  Wait for boot + configure SIM
# ─────────────────────────────────────────────
import subprocess, os, time

ADB  = f'{os.environ.get("ANDROID_HOME", "/opt/android-sdk")}/platform-tools/adb'
NET  = os.environ.get('LEGION_NET_TYPE', 'lte')
SIG  = os.environ.get('LEGION_SIGNAL', '4')

def adb(cmd, capture=False):
    r = subprocess.run(f'{ADB} {cmd}', shell=True, capture_output=capture,
                       text=True, env=os.environ)
    return r

def emu(cmd):
    """Send a command to the emulator console via socat."""
    return subprocess.run(
        f'echo -e "{cmd}\\nquit" | socat - TCP:localhost:5554',
        shell=True, capture_output=True, text=True
    )

print('Waiting for Android 15 to finish booting...')
max_wait = 480  # Android 15 may need longer on Colab
interval = 10
elapsed  = 0
booted   = False

while elapsed < max_wait:
    r = adb('shell getprop sys.boot_completed', capture=True)
    if r.stdout.strip() == '1':
        booted = True
        break
    elapsed += interval
    print(f'  [{elapsed}s] booting...')
    time.sleep(interval)

if not booted:
    print('TIMEOUT: emulator did not boot in 8 minutes.')
    print('Run Cell 15 to check the log.')
else:
    print()
    print('=' * 50)
    print('  Android 15 is ready.')
    print('=' * 50)

    adb('devices')

    r = adb('shell getprop ro.build.version.release', capture=True)
    print(f'Android version : {r.stdout.strip()}')

    r = adb('shell getprop ro.product.model', capture=True)
    print(f'Device model    : {r.stdout.strip()}')

    r = adb('shell getprop ro.build.version.sdk', capture=True)
    print(f'API level       : {r.stdout.strip()}')

    # Configure SIM/network via emulator console
    time.sleep(3)
    print()
    print('Configuring SIM card and network...')

    # Set network type
    r = emu(f'network lte')
    print(f'  Network type set to LTE')

    # Set signal strength
    r = emu(f'gsm signal {SIG} {SIG}')
    print(f'  Signal strength set to {SIG}/4')

    # Mark SIM as present
    r = emu('gsm data home')
    print(f'  GSM data: home')

    r = emu('gsm voice home')
    print(f'  GSM voice: home')

    # Dismiss lock screen
    time.sleep(2)
    adb('shell input keyevent 82')
    adb('shell input keyevent 4')
    print()
    print('Lock screen dismissed.')
    print('Open the URL from Cell 9 in your browser.')
    print()
    print('SIM card is active. Run Cell 16 for telephony commands.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 11  ADB helpers
# ─────────────────────────────────────────────
import subprocess, os

ADB = f'{os.environ.get("ANDROID_HOME", "/opt/android-sdk")}/platform-tools/adb'

def adb(cmd):
    r = subprocess.run(f'{ADB} {cmd}', shell=True, capture_output=True, text=True, env=os.environ)
    if r.stdout: print(r.stdout.strip())
    if r.stderr: print('[err]', r.stderr.strip())
    return r

adb('devices')

# Uncomment what you need:
# adb('shell screencap /sdcard/screen.png && pull /sdcard/screen.png /tmp/screen.png')
# adb('install /content/your_app.apk')
# adb('shell am start -a android.intent.action.VIEW -d "https://google.com"')
# adb('shell input tap 540 960')
# adb('shell input text "hello world"')
# adb('shell input swipe 540 2000 540 800 300')
# adb('shell pm list packages')   # list installed apps

print('ADB ready.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 12  Screenshot
# ─────────────────────────────────────────────
import subprocess, os
from IPython.display import Image, display

ADB  = f'{os.environ.get("ANDROID_HOME", "/opt/android-sdk")}/platform-tools/adb'
DEST = '/tmp/legion_screen.png'

subprocess.run(
    f'{ADB} shell screencap /sdcard/screen.png && {ADB} pull /sdcard/screen.png {DEST}',
    shell=True, env=os.environ, capture_output=True
)

if os.path.exists(DEST):
    display(Image(DEST, width=360))
else:
    print('Screenshot not found. Is the emulator booted?')

In [ ]:
# ─────────────────────────────────────────────
# CELL 13  Install APK from URL
# ─────────────────────────────────────────────
import subprocess, os

ADB     = f'{os.environ.get("ANDROID_HOME", "/opt/android-sdk")}/platform-tools/adb'
APK_URL = 'https://example.com/your-app.apk'  # replace with real URL
APK_TMP = '/tmp/legion_install.apk'

# Uncomment to run:
# subprocess.run(f'wget -q "{APK_URL}" -O {APK_TMP}', shell=True, check=True)
# r = subprocess.run(f'{ADB} install -r {APK_TMP}', shell=True, env=os.environ,
#                    capture_output=True, text=True)
# print('Installed.' if r.returncode == 0 else f'Failed: {r.stderr}')

print('Set APK_URL and uncomment the lines above.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 14  Screen record (save to /tmp/legion_record.mp4)
# ─────────────────────────────────────────────
import subprocess, os, time, threading

ADB     = f'{os.environ.get("ANDROID_HOME", "/opt/android-sdk")}/platform-tools/adb'
DEST    = '/tmp/legion_record.mp4'
RECORD_SECONDS = 10   # how many seconds to record

# Uncomment to run:
# proc = subprocess.Popen(
#     f'{ADB} shell screenrecord --time-limit {RECORD_SECONDS} /sdcard/legion_record.mp4',
#     shell=True, env=os.environ
# )
# proc.wait()
# subprocess.run(f'{ADB} pull /sdcard/legion_record.mp4 {DEST}', shell=True, env=os.environ)
# print(f'Saved: {DEST}')

print(f'Set RECORD_SECONDS and uncomment the lines above. Max 180s per Android limit.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 15  View logs
# ─────────────────────────────────────────────
import subprocess

print('=== Emulator log (last 80 lines) ===')
subprocess.run('tail -80 /tmp/emulator.log', shell=True)

print('\n=== noVNC log ===')
subprocess.run('tail -20 /tmp/novnc.log', shell=True)

print('\n=== x11vnc log ===')
subprocess.run('tail -20 /tmp/x11vnc.log', shell=True)

In [ ]:
# ─────────────────────────────────────────────
# CELL 16  SIM card / Telephony helpers
# ─────────────────────────────────────────────
# All telephony commands go through the emulator console on port 5554.
# Use socat to send commands, or the 'adb emu' shortcut.

import subprocess, os

ADB = f'{os.environ.get("ANDROID_HOME", "/opt/android-sdk")}/platform-tools/adb'

def emu_cmd(cmd):
    """Send a command to the emulator console."""
    r = subprocess.run(
        f'echo -e "{cmd}\\nquit" | socat - TCP:localhost:5554',
        shell=True, capture_output=True, text=True
    )
    out = r.stdout.strip()
    if out: print(out)
    return r

# ── Quick SIM status ──────────────────────────────────────
print('=== SIM / Telephony status ===')
r = subprocess.run(f'{ADB} shell dumpsys telephony.registry',
                   shell=True, capture_output=True, text=True, env=os.environ)
# Print the interesting lines only
for line in r.stdout.split('\n'):
    if any(k in line for k in ['mPhoneNumber', 'mNetworkType', 'mDataConnectionState',
                                 'mSignalStrength', 'mSimState', 'mOperatorAlpha']):
        print(line.strip())

print()
print('── Available commands (uncomment to run) ──')

# ── Simulate incoming SMS ─────────────────────────────────
# emu_cmd('sms send +15559876543 "Hello from Legion Sandbox"')

# ── Simulate incoming call ────────────────────────────────
# emu_cmd('gsm call +15559876543')

# ── Accept an incoming call ───────────────────────────────
# emu_cmd('gsm accept +15559876543')

# ── End/cancel a call ─────────────────────────────────────
# emu_cmd('gsm cancel +15559876543')

# ── Set signal strength (0-4) ─────────────────────────────
# emu_cmd('gsm signal 4 4')   # rssi=4, ber=4

# ── Change network type ───────────────────────────────────
# Options: gsm, edge, gprs, hsdpa, lte, 5gnr, none
# emu_cmd('network lte')

# ── Simulate no signal ────────────────────────────────────
# emu_cmd('gsm signal 0 0')
# emu_cmd('gsm voice unregistered')

# ── Restore signal ────────────────────────────────────────
# emu_cmd('gsm signal 4 4')
# emu_cmd('gsm voice home')
# emu_cmd('gsm data home')

# ── Check connected devices ───────────────────────────────
# subprocess.run(f'{ADB} devices', shell=True, env=os.environ)

print('\nUncomment any block above and re-run this cell.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 17  Keep-alive (prevents Colab idle timeout)
# ─────────────────────────────────────────────
# Press ■ to stop.
import time

print('Keeping session alive. Press ■ to stop.')
count = 0
while True:
    count += 1
    print(f'  [{count * 60}s] alive', flush=True)
    time.sleep(60)